## Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import random
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Hyperparameter Setting

In [ ]:
CFG = {
    'TRAIN_WINDOW_SIZE':90,
    'PREDICT_SIZE':21, # 21일치 예측
    'EPOCHS':10+5,
    'LEARNING_RATE':1e-4,
    'BATCH_SIZE':4096,
    'SEED':493
}

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['SEED']) # Seed 고정

### 데이터 불러오기

In [ ]:
train_data = pd.read_csv('./train.csv').drop(columns=['ID', '제품'])

# need to scale
price_change_rate = pd.read_csv('./price_change_rate.csv').drop(columns=['ID', '제품'])
train_change_rate = pd.read_csv('./train_change_rate.csv').drop(columns=['ID', '제품'])

# no need to scale
weekday_data = pd.read_csv('./weekday_data.csv').drop(columns=['ID', '제품'])
holiday_data = pd.read_csv('./holiday_data.csv').drop(columns=['ID', '제품'])
함께사용_data = pd.read_csv('./함께사용.csv')
소비주기_data = pd.read_csv('./소비주기.csv')

### 데이터 전처리

In [ ]:
# Data Scaling
maxi = train_data.iloc[:,4:].max(axis=1)
mini = train_data.iloc[:,4:].min(axis=1)

scale_max_dict = maxi.to_dict()
scale_min_dict = mini.to_dict()

scaled_data = (train_data.iloc[:,4:] - mini[:, None]) / (maxi - mini)[:, None]
scaled_data[maxi == mini] = 0

# Scaled data를 원래 데이터프레임에 업데이트
train_data.update(scaled_data)

In [ ]:
def data_scaling_2(data):
    maximum = data.iloc[:,4:].max(axis=1)
    minimum = data.iloc[:,4:].min(axis=1)

    max_dict = maximum.to_dict()
    min_dict = minimum.to_dict()

    scaled_d = (data.iloc[:,4:] - minimum[:, None]) / (maximum - minimum)[:, None]
    scaled_d[maximum == minimum] = 0
           #[maxi == minimum] 였음
    # Scaled data를 원래 데이터프레임에 업데이트
    data.update(scaled_d)

    return data

In [ ]:
data_scaling_2(price_change_rate)
data_scaling_2(train_change_rate)
print('Done.')

Done.


In [ ]:
# def label_encoding(data):
#     label_encoder = LabelEncoder()
#     categorical_columns = ['대분류', '중분류', '소분류', '브랜드']

#     for col in categorical_columns:
#         label_encoder.fit(data[col])
#         data[col] = label_encoder.transform(data[col])

#     return data

In [ ]:
# label_encoding(train_data)
# label_encoding(price_change_rate)
# label_encoding(weekday_data)
# label_encoding(holiday_data)
# print('Done.')

In [ ]:
train_data.drop(['대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True) # 4개의 상태 특성을 뺌.
price_change_rate.drop(['대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True)
train_change_rate.drop(['대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True)
weekday_data.drop(['대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True)
holiday_data.drop(['대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True)

# **##########################################################################**

In [ ]:
(len(train_data.columns) - 111 + 1)

349

In [ ]:
# make_train_data
data = train_data
train_size=CFG['TRAIN_WINDOW_SIZE']
predict_size=CFG['PREDICT_SIZE']
'''
학습 기간 블럭, 예측 기간 블럭의 세트로 데이터를 생성
data : 일별 판매량
train_size : 학습에 활용할 기간
predict_size : 추론할 기간
'''
num_rows = len(data)
window_size = train_size + predict_size

train_input = np.empty((num_rows * (len(data.columns) - window_size + 1), train_size, len(data.iloc[0, :4]) + 1 - 4 + 6)) # - (브랜드, 대,중,소분류) + 추가할 특성의 개수
train_target = np.empty((num_rows * (len(data.columns) - window_size + 1), predict_size))

for i in tqdm(range(num_rows)):
    sales_data = np.array(data.iloc[i, 4:]) # 판매량 데이터 train
    ######################################################################## 기존 코드에서 추가되는 특성들
    price_change_rate_array = np.array(price_change_rate.iloc[i, 4:])
    train_change_rate_array = np.array(train_change_rate.iloc[i, 4:])
    weekday_array = np.array(weekday_data.iloc[i, 4:])
    holiday_array = np.array(holiday_data.iloc[i, 4:])
    함께사용_array = np.array(함께사용_data.iloc[i, 4:])
    소비주기_array = np.array(소비주기_data.iloc[i, 4:])
    ########################################################################
    ########################################################################
    for j in range(len(sales_data) - window_size + 1):
        window = sales_data[j : j + window_size]
    ########################################################################
        window_2 = price_change_rate_array[j : j + window_size]
        window_3 = train_change_rate_array[j : j + window_size]
        window_4 = weekday_array[j : j + window_size]
        window_5 = holiday_array[j : j + window_size]
        window_6 = 함께사용_array[j : j + window_size]
        window_7 = 소비주기_array[j : j + window_size]
    ########################################################################

        temp_data = np.column_stack((window[:train_size], window_2[:train_size], window_3[:train_size], window_4[:train_size], window_5[:train_size], window_6[:train_size], window_7[:train_size]))

        train_input[i * (len(data.columns) - window_size + 1) + j] = temp_data
        train_target[i * (len(data.columns) - window_size + 1) + j] = window[train_size:]

  0%|          | 0/15890 [00:00<?, ?it/s]

# **###########################################**

In [ ]:
# make_test_data
data = train_data
train_size=CFG['TRAIN_WINDOW_SIZE']
'''
평가 데이터(Test Dataset)를 추론하기 위한 Input 데이터를 생성
data : 일별 판매량
train_size : 추론을 위해 필요한 일별 판매량 기간 (= 학습에 활용할 기간)
'''
num_rows = len(data)

test_input = np.empty((num_rows, train_size, len(data.iloc[0, :4]) + 1 - 4 + 6)) # + 추가할 특성의 개수

for i in tqdm(range(num_rows)):
    sales_data = np.array(data.iloc[i, -train_size:])
    ######################################################################## 기존 코드에서 추가되는 특성들
    price_change_rate_array = np.array(price_change_rate.iloc[i, -train_size:])
    train_change_rate_array = np.array(train_change_rate.iloc[i, -train_size:])
    weekday_array = np.array(weekday_data.iloc[i, -train_size:])
    holiday_array = np.array(holiday_data.iloc[i, -train_size:])
    함께사용_array = np.array(함께사용_data.iloc[i, -train_size:])
    소비주기_array = np.array(소비주기_data.iloc[i, -train_size:])
    ########################################################################
    ########################################################################
    window_8 = sales_data[-train_size : ]
    ########################################################################
    window_9 = price_change_rate_array[-train_size : ]
    window_10 = train_change_rate_array[-train_size : ]
    window_11 = weekday_array[-train_size : ]
    window_12 = holiday_array[-train_size : ]
    window_13 = 함께사용_array[-train_size : ]
    window_14 = 소비주기_array[-train_size : ]
    ########################################################################

    temp_data_2 = np.column_stack((window_8[:train_size], window_9[:train_size], window_10[:train_size], window_11[:train_size], window_12[:train_size], window_13[:train_size], window_14[:train_size]))
    test_input[i] = temp_data_2

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
train_input.shape, train_target.shape, test_input.shape

((5545610, 90, 7), (5545610, 21), (15890, 90, 7))

In [ ]:
# train_input, train_target = make_train_data(train_data)
# test_input = make_predict_data(train_data)

# **##########################################################################**

In [ ]:
# Train / Validation Split
data_len = len(train_input)
val_input = train_input[-int(data_len*0.2):]
val_target = train_target[-int(data_len*0.2):]
train_input = train_input[:-int(data_len*0.2)]
train_target = train_target[:-int(data_len*0.2)]

In [ ]:
train_input.shape, train_target.shape, val_input.shape, val_target.shape, test_input.shape

((4436488, 90, 7),
 (4436488, 21),
 (1109122, 90, 7),
 (1109122, 21),
 (15890, 90, 7))

### Custom Dataset

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __getitem__(self, index):
        if self.Y is not None:
            return torch.Tensor(self.X[index]), torch.Tensor(self.Y[index])
        return torch.Tensor(self.X[index])

    def __len__(self):
        return len(self.X)

In [ ]:
train_dataset = CustomDataset(train_input, train_target)
train_loader = DataLoader(train_dataset, batch_size = CFG['BATCH_SIZE'], shuffle=True, num_workers=0)

val_dataset = CustomDataset(val_input, val_target)
val_loader = DataLoader(val_dataset, batch_size = CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

### 모델 선언

In [ ]:
class BaseModel(nn.Module):
    def __init__(self, input_size=7, hidden_size=512, output_size=CFG['PREDICT_SIZE']): # input_size 변경할 것
        super(BaseModel, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size//2),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(hidden_size//2, output_size)
        )

        self.actv = nn.ReLU()

    def forward(self, x):
        # x shape: (B, TRAIN_WINDOW_SIZE, 5)
        batch_size = x.size(0)
        hidden = self.init_hidden(batch_size, x.device)

        # LSTM layer
        lstm_out, hidden = self.lstm(x, hidden)

        # Only use the last output sequence
        last_output = lstm_out[:, -1, :]

        # Fully connected layer
        output = self.actv(self.fc(last_output))

        return output.squeeze(1)

    def init_hidden(self, batch_size, device):
        # Initialize hidden state and cell state
        return (torch.zeros(1, batch_size, self.hidden_size, device=device),
                torch.zeros(1, batch_size, self.hidden_size, device=device))

### 모델 학습

In [ ]:
def train(model, optimizer, train_loader, val_loader, device):
    model.to(device)
    criterion = nn.MSELoss().to(device)
    best_loss = 9999999
    best_model = None

    for epoch in range(1, CFG['EPOCHS']+1):
        model.train()
        train_loss = []
        train_mae = []
        for X, Y in tqdm(iter(train_loader)):
            X = X.to(device)
            Y = Y.to(device)

            optimizer.zero_grad()

            output = model(X)
            loss = criterion(output, Y)

            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())

        val_loss = validation(model, val_loader, criterion, device)
        print(f'Epoch : [{epoch}] Train Loss : [{np.mean(train_loss):.5f}] Val Loss : [{val_loss:.5f}]')

        if best_loss > val_loss:
            best_loss = val_loss
            best_model = model
            print('Model Saved')
    return best_model

In [ ]:
def validation(model, val_loader, criterion, device):
    model.eval()
    val_loss = []

    with torch.no_grad():
        for X, Y in tqdm(iter(val_loader)):
            X = X.to(device)
            Y = Y.to(device)

            output = model(X)
            loss = criterion(output, Y)

            val_loss.append(loss.item())
    return np.mean(val_loss)

## Run !!

In [ ]:
model = BaseModel()
optimizer = torch.optim.Adam(params = model.parameters(), lr = CFG["LEARNING_RATE"])
infer_model = train(model, optimizer, train_loader, val_loader, device)

  0%|          | 0/1084 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

Epoch : [1] Train Loss : [0.01967] Val Loss : [0.01760]
Model Saved


  0%|          | 0/1084 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

...

  0%|          | 0/271 [00:00<?, ?it/s]

Epoch : [14] Train Loss : [0.01594] Val Loss : [0.01584]


  0%|          | 0/1084 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

Epoch : [15] Train Loss : [0.01590] Val Loss : [0.01573]
Model Saved


## 모델 추론

In [ ]:
test_dataset = CustomDataset(test_input, None)
test_loader = DataLoader(test_dataset, batch_size = CFG['BATCH_SIZE'], shuffle=False, num_workers=0)

In [ ]:
def inference(model, test_loader, device):
    predictions = []

    with torch.no_grad():
        for X in tqdm(iter(test_loader)):
            X = X.to(device)

            output = model(X)

            # 모델 출력인 output을 CPU로 이동하고 numpy 배열로 변환
            output = output.cpu().numpy()

            predictions.extend(output)

    return np.array(predictions)

In [ ]:
pred = inference(infer_model, test_loader, device)

  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# 추론 결과를 inverse scaling
for idx in range(len(pred)):
    pred[idx, :] = pred[idx, :] * (scale_max_dict[idx] - scale_min_dict[idx]) + scale_min_dict[idx]

# 결과 후처리
pred = np.round(pred, 0).astype(int)

In [ ]:
pred.shape

(15890, 21)

# 모델 저장

In [ ]:
# 모델의 가중치와 구조를 저장합니다.
torch.save(infer_model.state_dict(), './54744.pth')  # 가중치 저장
# 모델 아키텍처를 저장하고 싶을 때
torch.save(infer_model, './54744_with_architecture.pth')  # 구조와 가중치 함께 저장

# # 나중에 모델을 불러올 때 사용합니다.
# model = infer_model()  # 모델 아키텍처를 생성
# model.load_state_dict(torch.load('my_model.pth'))  # 가중치 불러오기
# # 구조와 가중치 함께 저장한 경우 불러오는 방법
# loaded_model = torch.load('my_model_with_architecture.pth')  # 구조와 가중치 함께 불러오기

## Submission

In [ ]:
submit = pd.read_csv('./sample_submission.csv')
# submit.head()

In [ ]:
submit.iloc[:,1:] = pred
submit.head()

,ID,2023-04-05,2023-04-06,2023-04-07,2023-04-08,2023-04-09,2023-04-10,2023-04-11,2023-04-12,2023-04-13,...,2023-04-16,2023-04-17,2023-04-18,2023-04-19,2023-04-20,2023-04-21,2023-04-22,2023-04-23,2023-04-24,2023-04-25
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2,2,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,1
3,3,0,0,0,0,0,0,0,0,1,...,1,1,1,2,2,2,2,2,2,2
4,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,1,1,1,1


In [ ]:
submit.to_csv('./54744.csv', index=False)

# 정보

In [ ]:
import sys

print("-------------------------- Python & library version --------------------------")
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("numpy version: {}".format(np.__version__))
print("torch version: {}".format(torch.__version__))
print("------------------------------------------------------------------------------")

-------------------------- Python & library version --------------------------
Python version: 3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]
pandas version: 1.5.3
numpy version: 1.23.5
torch version: 2.0.1+cu118
------------------------------------------------------------------------------


In [ ]:
import platform
import psutil

# 운영체제 정보
print("운영체제:", platform.system())
print("운영체제 버전:", platform.version())

# CPU 정보
with open('/proc/cpuinfo', 'r') as f:
    cpu_info = f.readlines()

cpu_model = [x.strip() for x in cpu_info if 'model name' in x][0].split(':')[1].strip()
cpu_cores = psutil.cpu_count(logical=False)
print("프로세서 모델:", cpu_model)
print("물리적인 CPU 코어 수:", cpu_cores)

# 메모리 정보
memory = psutil.virtual_memory()
total_memory_gb = round(memory.total / (1024 ** 3))
available_memory_gb = round(memory.available / (1024 ** 3))
print("총 메모리:", total_memory_gb, "GB")
print("사용 가능한 메모리:", available_memory_gb, "GB")

# GPU 정보
try:
    gpu_info = !nvidia-smi
    print("\n".join(gpu_info))
except Exception as e:
    print("GPU 정보를 확인할 수 없습니다.")

운영체제: Linux
운영체제 버전: #1 SMP Fri Jun 9 10:57:30 UTC 2023
프로세서 모델: Intel(R) Xeon(R) CPU @ 2.20GHz
물리적인 CPU 코어 수: 4
총 메모리: 51 GB
사용 가능한 메모리: 20 GB
Sun Aug 27 17:02:03 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-SXM2...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   40C    P0    39W / 300W |  11454MiB / 16384MiB |      0%      Default |
|                               |                      |                  N/A |
+-----------------------